# Kateto 1.5B v3-bis — SFT (ctx 512) + style tune del estado S_0 + apply (2xT4)

Objetivo: un 1.5B coherente **con la voz de Kateto adentro** y un unico `.pth`
que el camino nativo sabe cargar.

Pipeline (K3):
1. SFT Capa 1 (PiSSA r=32 sobre el 1.5B, ctx 512, micro 8, bf16, 3 epocas) -> `out/rwkv_kateto_sft/`
2. **Style tune (K3)**: `--peft state` sobre el modelo SFTeado -> entrena SOLO
   `blocks.{i}.att.time_state` (el 100% de los pesos queda congelado). Voz `seco`.
   Deja `loss_data.jsonl` + `out/rwkv_states/seco/rwkv-{0..4}.pth`.
3. **apply** (`scripts/style_tune_kateto.py`): cose el head de estilo sobre el base
   -> `/kaggle/working/kateto-ckpt/kateto-1.5b-seco.pth` (el `.pth` final).

Dataset: `gabrielsolotorevsky/kateto-sft` (corpus plus, 55137 muestras).


In [ ]:
# CELL 1 — Setup: env, deps, GPU probe
import os
os.environ["NCCL_P2P_DISABLE"] = "1"   # 2xT4 sin NVLink: DDP sobre PCIe
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["RWKV_MY_TESTING"] = "x070"
os.environ["WKV"] = "fla"

!pip uninstall -y torchao
!pip install -q --no-cache-dir rwkv rwkv-fla einops jsonlines peft datasets triton huggingface_hub "lightning>=2.0.0"
!nvidia-smi

In [ ]:
# CELL 2 — Modelo base 1.5B (3 GB desde HF)
import os, shutil, torch
from huggingface_hub import hf_hub_download

MODEL_REPO = "BlinkDL/rwkv7-g1"
MODEL_FILE = "rwkv7-g1j-1.5b-20260831-ctx16384.pth"
N_LAYER, N_EMBD = 24, 2048

os.makedirs("/kaggle/working/models", exist_ok=True)
model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir="/kaggle/working/models")
if os.path.islink(model_path):
    target = os.path.realpath(model_path)
    os.remove(model_path)
    shutil.copy2(target, model_path)
shutil.rmtree(os.path.expanduser("~/.cache/huggingface"), ignore_errors=True)
print("base:", model_path, os.path.getsize(model_path), "bytes")
print("GPUs:", torch.cuda.device_count())

In [ ]:
# CELL 3 — Stage del dataset desde /kaggle/input (layout agnostico)
import os, glob, shutil, zipfile, tarfile

INPUT = "/kaggle/input"
print("arbol de /kaggle/input:")
vistos = 0
for root, dirs, files in os.walk(INPUT, followlinks=True):
    prof = root.replace(INPUT, "").count("/")
    if prof > 3:
        dirs[:] = []
        continue
    for n in sorted(files)[:8]:
        print("   ", os.path.join(root, n))
        vistos += 1
    for d in sorted(dirs)[:8]:
        print("   ", os.path.join(root, d) + "/")
    if vistos > 60:
        break

zips = sorted(glob.glob(f"{INPUT}/**/*.zip", recursive=True))
tars = sorted(glob.glob(f"{INPUT}/**/*.tar", recursive=True))
plus = sorted(glob.glob(f"{INPUT}/**/rwkv_kateto_base_train_plus.jsonl", recursive=True))
pefts = sorted(glob.glob(f"{INPUT}/**/RWKV-PEFT/train.py", recursive=True))
print("zips:", zips[:10])
print("tars:", tars[:10])
print("plus jsonl:", plus[:3])
print("RWKV-PEFT/train.py:", pefts[:3])
assert plus and pefts, "no encontre el dataset de Kateto (plus + RWKV-PEFT) montado en /kaggle/input"

TARGET = "/kaggle/working/kateto-train"
os.makedirs(TARGET, exist_ok=True)

for z in zips:
    base = os.path.basename(z)[:-4]
    with zipfile.ZipFile(z) as zf:
        names = zf.namelist()
    top = names[0].split("/")[0] if names else ""
    dest = TARGET if top == base else os.path.join(TARGET, base)
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(dest)
    print(f"unzip {os.path.basename(z)} -> {dest} ({len(names)} entradas)")

# Kaggle sube las carpetas como .tar (--dir-mode tar): las descomprime aca.
for t in tars:
    with tarfile.open(t) as tf:
        names = tf.getnames()
    top = names[0].split("/")[0] if names else ""
    dest = TARGET if top == os.path.basename(t)[:-4] else os.path.join(TARGET, top)
    os.makedirs(dest, exist_ok=True)
    with tarfile.open(t) as tf:
        tf.extractall(dest if top == os.path.basename(t)[:-4] else TARGET)
    print(f"untar {os.path.basename(t)} -> {dest} ({len(names)} entradas)")

# carpetas sueltas (Kaggle ya descomprime los tar/zip de los datasets)
if not os.path.exists(f"{TARGET}/RWKV-PEFT/train.py"):
    srcp = os.path.dirname(os.path.dirname(pefts[0])) if pefts else None
    if srcp and os.path.exists(os.path.join(srcp, "data")):
        print("copiando desde:", srcp)
        for p in glob.glob(f"{srcp}/**/*", recursive=True):
            if os.path.isfile(p):
                rel = os.path.relpath(p, srcp)
                dst = os.path.join(TARGET, rel)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                if not os.path.exists(dst):
                    shutil.copy2(p, dst)

os.makedirs("/kaggle/working/out", exist_ok=True)
if not os.path.exists(f"{TARGET}/out"):
    os.symlink("/kaggle/working/out", f"{TARGET}/out")

for req in ["RWKV-PEFT/train.py", "data/rwkv_kateto_base_train_plus.jsonl",
            "data/eval_prompts.jsonl", "data/rwkv_kateto_seco.jsonl",
            "rwkv_pipeline/infer_kateto.py", "rwkv_pipeline/train_state_voice.sh",
            "scripts/style_tune_kateto.py", "RWKV-PEFT/rwkvt/dataset/dataset.py",
            "RWKV-PEFT/rwkvt/lightning_train/trainer.py"]:
    assert os.path.exists(f"{TARGET}/{req}"), f"falta {req}"
    print("ok:", req)
print("jsonl en data/:", len(glob.glob(f"{TARGET}/data/*.jsonl")))


In [ ]:
# CELL 4 — Sanidad del corpus: formato exacto del train set plus (55137 muestras)
import json, re

TARGET = "/kaggle/working/kateto-train"
PLUS = f"{TARGET}/data/rwkv_kateto_base_train_plus.jsonl"
TURNO = re.compile(r"^(?:<\|im_user\|>[^<]*(?:<(?!\|im_end\|>)[^<]*)*<\|im_end\|>\n"
                   r"<\|im_start\|>[A-Za-z0-9_]+\n[^<]*(?:<(?!\|im_end\|>)[^<]*)*<\|im_end\|>\n?)+$")

ok = mal = 0
malas = []
for line in open(PLUS, encoding="utf-8"):
    t = json.loads(line)["text"]
    if TURNO.match(t):
        ok += 1
    else:
        mal += 1
        if len(malas) < 3:
            malas.append(t[:200])
print(f"train plus: {ok} bien formadas, {mal} mal formadas")
assert mal == 0, f"hay muestras mal formadas: {malas}"
print("eval prompts:", sum(1 for _ in open(f"{TARGET}/data/eval_prompts.jsonl", encoding="utf-8")))


In [ ]:
# CELL 5 — SFT Capa 1: 1.5B, ctx 512, DDP 2xT4, epocas completas
# 4526 pasos x 2,31 s = 2,9 h por epoca, 3 epocas = 8,7 h, guarda cada 2,9 h
import os, subprocess, time, torch, glob

TARGET = "/kaggle/working/kateto-train"
CWD = f"{TARGET}/RWKV-PEFT"
PROJ = "/kaggle/working/out/rwkv_kateto_sft"
os.makedirs(PROJ, exist_ok=True)

EPOCH_STEPS = 700   # 700 x 8 = 5600 muestras por epoca; ~2,8 h por checkpoint a 15,9 s/paso (medido en la corrida real bf16+merge)
EPOCHS = 3          # 2100 pasos x 15,9 s = 9,3 h: entra en las 12 h con margen para el export
CTX = 512
MICRO_BSZ = 8

n_gpu = torch.cuda.device_count()
devices, strategy = (str(n_gpu), "ddp") if n_gpu > 1 else ("1", "auto")

cmd = ["python", "train.py",
       "--load_model", "/kaggle/working/models/rwkv7-g1j-1.5b-20260831-ctx16384.pth",
       "--proj_dir", PROJ,
       "--data_file", f"{TARGET}/data/rwkv_kateto_base_train_plus.jsonl",
       "--data_type", "jsonl", "--loss_mask", "qa", "--vocab_size", "65536",
       "--n_layer", "24", "--n_embd", "2048",
       "--ctx_len", str(CTX), "--micro_bsz", str(MICRO_BSZ),
       "--epoch_steps", str(EPOCH_STEPS), "--epoch_count", str(EPOCHS), "--epoch_save", "1",
       "--lr_init", "2e-4", "--lr_final", "2e-5",
       "--accelerator", "gpu", "--devices", devices, "--strategy", strategy,
       "--precision", "bf16",
       # fp16 muere en este camino: el kernel Triton (op fla) mezcla fp16 y fp32 -> CompilationError
       "--grad_cp", "1",
       "--num_workers", "2",
       "--my_testing", "x070", "--op", "fla",
       "--peft", "pissa",
       "--pissa_config", '{"pissa_load":"","pissa_init":"","pissa_r":32,"svd_niter":4}',
       "--train_parts", '["time","ln"]', "--merge", "1"]

log = "/kaggle/working/out/sft.log"
print(">>>", " ".join(cmd[:8]), "... (log en", log, ")")
t0 = time.time()
with open(log, "w") as fh:
    r = subprocess.run(cmd, cwd=CWD, stdout=fh, stderr=subprocess.STDOUT)
print(f"SFT rc={r.returncode} en {(time.time()-t0)/3600:.2f} h")
print(open(log).read()[-2500:])
print("ckpts:", sorted(glob.glob(f"{PROJ}/*.pth")))


In [ ]:
# CELL 6 — ORPO: DESHABILITADO (implementacion vendoreada con defectos medidos)
#   - off-by-one: con pares mas largos que ctx_len+1 el forward asserta
#     "Cannot forward, model ctx_len is exhausted".
#   - nan: el termino de razon de odds se va a nan en el paso 2 (medido 2026-09-16 y 24).
# Si en el futuro se arreglan, esta es la celda donde va: data_type orpo, ctx 512,
# micro_bsz 2, epoch_steps 4000, orpo_lambda 0.1, merge 1.
print("ORPO omitido: no hay implementacion estable en el repo todavia.")

In [ ]:
# CELL 7 — Estado de voz (Capa 2) del 1.5B: SOLO `seco`
# Es la voz que usa el motor por defecto (--voice seco) y la que usa la app.
# Las otras (doktor/jane/whisperer/streamer) son 3x mas epocas cada una y no entran
# en el presupuesto de 12 h junto con el SFT: van en otra corrida.
import os, subprocess, glob, time

TARGET = "/kaggle/working/kateto-train"
cands = sorted(glob.glob("/kaggle/working/out/rwkv_kateto_sft/*.pth"), key=os.path.getmtime)
assert cands, "sin checkpoint de Capa 1"
FINAL = cands[-1]
print("modelo final para el estado:", FINAL)

os.makedirs("/kaggle/working/out/rwkv_kateto_base", exist_ok=True)
dst = "/kaggle/working/out/rwkv_kateto_base/rwkv-final.pth"
if not os.path.exists(dst):
    os.link(FINAL, dst)

env = os.environ.copy()
env["N_LAYER"], env["N_EMBD"] = "24", "2048"     # el script adivina por path: se lo fijamos
env["MICRO_BSZ"], env["GRAD_CP"] = "4", "0"
env["DEVICES"], env["STRATEGY"] = "1", "auto"
env["PROJECT"] = TARGET

t0 = time.time()
r = subprocess.run(["bash", "rwkv_pipeline/train_state_voice.sh", "seco", dst], cwd=TARGET, env=env)
print(f"seco: rc={r.returncode} en {(time.time()-t0)/60:.1f} min")
for d in sorted(glob.glob("/kaggle/working/out/rwkv_states/*")):
    print(d, sorted(os.path.basename(x) for x in glob.glob(f"{d}/*.pth"))[-3:])

In [ ]:
import json
# CELL 8 — apply (K3): cose el head de estilo (time_state) sobre el base -> .pth final
# El head slim que deja `--peft state` son SOLO los tensores blocks.{i}.att.time_state.
# apply lo escribe sobre el checkpoint SFTeado y produce un unico .pth que el camino
# nativo carga (rwkv_pipeline/infer_kateto.py busca exactamente esas claves).
import os, subprocess, glob

TARGET = "/kaggle/working/kateto-train"
BASE = "/kaggle/working/out/rwkv_kateto_base/rwkv-final.pth"
heads = sorted(glob.glob("/kaggle/working/out/rwkv_states/seco/rwkv-*.pth"), key=os.path.getmtime)
assert heads, "no hay head de estilo: la celda del style tune no dejo rwkv-*.pth"
HEAD = heads[-1]
print("base:", BASE, "| head de estilo:", HEAD)

os.makedirs("/kaggle/working/kateto-ckpt", exist_ok=True)
FINAL = "/kaggle/working/kateto-ckpt/kateto-1.5b-seco.pth"
for cmd in (["python3", "scripts/style_tune_kateto.py", "info", HEAD],
            ["python3", "scripts/style_tune_kateto.py", "apply", BASE,
             "--head", HEAD, "--out", FINAL]):
    print(">>>", " ".join(cmd))
    print("rc:", subprocess.run(cmd, cwd=TARGET).returncode)

assert os.path.exists(FINAL), "apply no escribio el .pth final"
print("FINAL:", FINAL, os.path.getsize(FINAL), "bytes")

# la senal real de entrenamiento (no el exit code): loss_data.jsonl de cada etapa
for p in sorted(glob.glob("/kaggle/working/out/**/loss_data.jsonl", recursive=True)):
    lns = [l for l in open(p).read().strip().splitlines() if l.strip()]
    losses = [json.loads(l)["loss"] for l in lns]
    if losses:
        print(f"{p}: {len(losses)} pasos | loss {losses[0]:.4f} -> {losses[-1]:.4f}")


In [ ]:
# CELL 8 — Eval de silencio y coherencia sobre el set fijo
import os, subprocess, json, glob

TARGET = "/kaggle/working/kateto-train"
cands = sorted(glob.glob("/kaggle/working/out/rwkv_kateto_sft/*.pth"), key=os.path.getmtime)
MODEL = cands[-1]
print("modelo a evaluar:", MODEL)

prompts = [json.loads(l)["prompt"] for l in open(f"{TARGET}/data/eval_prompts.jsonl", encoding="utf-8")]
env = os.environ.copy()
env["HSA_OVERRIDE_GFX_VERSION"] = "10.3.0"
env["PYTHONPATH"] = TARGET
resultados, silencios, cortas = [], 0, 0
for p in prompts:
    r = subprocess.run(["python", "rwkv_pipeline/infer_kateto.py", "--model", MODEL,
                        "--voice", "seco", "--prompt", p, "--allow_no_response"],
                       cwd=TARGET, env=env, capture_output=True, text=True, timeout=600)
    out = ""
    for line in r.stdout.splitlines():
        if line.startswith("[Kateto]:"):
            out = line.split("[Kateto]:", 1)[1].strip()
    es_sil = (out.strip() in ("<|no_response|>", "<|wait|>", "") )
    silencios += int(es_sil)
    cortas += int(len(out) < 20 and not es_sil)
    resultados.append({"prompt": p, "respuesta": out, "silencio": es_sil})
    print(f"— {p[:60]}\n   {out[:200]}\n", flush=True)

res = {"modelo": MODEL, "n": len(prompts), "silencios": silencios, "respuestas_cortas": cortas,
       "tasa_silencio": round(silencios / len(prompts), 3), "detalle": resultados}
json.dump(res, open("/kaggle/working/out/eval_v2.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps({k: v for k, v in res.items() if k != "detalle"}, indent=2))

In [ ]:
# CELL 9 — Export: checkpoint final + estados + eval
import os, glob, shutil, json

export = "/kaggle/working/kateto-ckpt"
os.makedirs(export, exist_ok=True)

cands = sorted(glob.glob("/kaggle/working/out/rwkv_kateto_sft/*.pth"), key=os.path.getmtime)
FINAL = cands[-1]
dst = os.path.join(export, "kateto-1.5b-sft.pth")
if not os.path.exists(dst):
    try:
        os.link(FINAL, dst)          # hardlink: mismo fs, no ocupa mas
    except OSError:
        shutil.copy2(FINAL, dst)
print("checkpoint exportado:", dst, os.path.getsize(dst), "bytes")

if os.path.exists("/kaggle/working/out/rwkv_states"):
    shutil.copytree("/kaggle/working/out/rwkv_states", f"{export}/rwkv_states", dirs_exist_ok=True)
for extra in ["eval_v2.json", "sft.log"]:
    p = f"/kaggle/working/out/{extra}"
    if os.path.exists(p):
        shutil.copy2(p, export)

# el base ya no hace falta
base = "/kaggle/working/models/rwkv7-g1j-1.5b-20260831-ctx16384.pth"
if os.path.exists(base):
    os.remove(base)
print("contenido de", export, ":", sorted(os.listdir(export)))
print("estados:", sorted(os.listdir(f"{export}/rwkv_states")) if os.path.exists(f"{export}/rwkv_states") else "ninguno")